# 3. Data statistics

*Author: Phạm Tấn Nhật Thịnh · code: `src/04_monitor.ipynb`, `src/05_run_pipeline.ipynb`*

This chapter describes what the crawl produced for the 20,000 assigned IDs: how many papers and versions
were scraped and what failed (3.1), how large a paper is before and after the figures are removed (3.2),
what the reference metadata looks like (3.3), and how the corpus breaks down by version count, subject and
date (3.4).

**Every number, table and figure below is computed when the book is built**, from the files in `data/`
that `tools/export_report_data.py` derives from the crawl's logs and state (the code of each cell can be
unfolded). The populations differ from section to section and are named each time:

| Level | Source | One row per |
|---|---|---|
| paper | `papers.csv` (pipeline log), `metadata.csv` (discovery manifest) | assigned ID |
| version | `versions.csv` (download receipts) | source version that was processed |
| request event | `download_events.csv` (`errors.jsonl`, `throttles.jsonl`) | failed or refused request |
| reference | `reference_fields.json`, `references_log.csv` | saved reference / logged request |

The four assigned ranges are called **R1–R4** in the order of `ID_RANGES` in `src/00_config.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import report_data as rd

rd.check()
rd.style()
bench = rd.benchmark()
papers, meta, versions, events = rd.papers(), rd.metadata(), rd.versions(), rd.events()
RANGE_LABEL = {name: f"{name}  {first}–{last}" for name, (first, last) in rd.RANGES.items()}


def pct(part, whole, digits=2):
    return f"{100 * part / whole:.{digits}f}%" if whole else "–"


def with_total(frame, label="Total"):
    total = {frame.columns[0]: label, **frame.sum(numeric_only=True).to_dict()}
    return pd.concat([frame, pd.DataFrame([total])], ignore_index=True)

## 3.1 Scraping success rate and failure breakdown

**Definition.** A paper is *scraped successfully* when **every one of its versions has a download
receipt**: the version was requested, unpacked and reduced to its `.tex`/`.bib` files, or it was found to
have no LaTeX source (PDF or Word only) and keeps an empty `vN/` folder, as the statement asks. A paper with
even one unprocessed version is counted as **incomplete**, although its other versions are in the corpus.

$$
\text{success rate} = \frac{\#\{\text{papers whose versions all have a receipt}\}}{\#\{\text{assigned IDs}\}}
$$

Reference metadata is evaluated separately in section 3.3 and does not enter this rate.

In [ ]:
by_range = papers.groupby("range").agg(
    assigned=("id", "size"),
    on_arxiv=("versions", lambda s: int((s > 0).sum())),
    scraped=("outcome", lambda s: int(s.isin(["latex", "mixed", "pdf_only"]).sum())),
    with_latex=("outcome", lambda s: int(s.isin(["latex", "mixed"]).sum())),
    pdf_only=("outcome", lambda s: int((s == "pdf_only").sum())),
    incomplete=("outcome", lambda s: int((s == "incomplete").sum())),
).reset_index()
by_range["range"] = by_range["range"].map(RANGE_LABEL)
t31 = with_total(by_range)
t31["success rate"] = 100 * t31["scraped"] / t31["assigned"]
t31.columns = ["Range", "Assigned IDs", "Found on arXiv", "Scraped successfully", "… with LaTeX source",
               "… without LaTeX (empty record)", "Incomplete", "Success rate"]
rd.table(t31, "Table 3.1. Paper-level outcome of the crawl by ID range.", {"Success rate": "{:.2f}%"})

total = t31.iloc[-1]
assert total["Scraped successfully"] == bench["papers_scraped"], "papers.csv and benchmark.json disagree"
rd.md(
    f"All **{int(total['Assigned IDs']):,}** assigned IDs exist on arXiv "
    f"({int(total['Assigned IDs'] - total['Found on arXiv']):,} unused IDs), so the success rate over the assigned "
    f"IDs and over the papers found is the same: **{total['Success rate']:.2f}%** "
    f"({int(total['Scraped successfully']):,} papers). **{int(total['… with LaTeX source']):,}** papers have LaTeX "
    f"for at least one version; **{int(total['… without LaTeX (empty record)']):,}** have none and are kept as empty "
    f"records; **{int(total['Incomplete']):,}** are incomplete. The rate is nearly the same in the four ranges "
    f"({t31['Success rate'][:-1].min():.2f}%–{t31['Success rate'][:-1].max():.2f}%).")

In [ ]:
order = ["latex", "mixed", "pdf_only", "incomplete"]
labels = {"latex": "LaTeX for every version", "mixed": "LaTeX for some versions",
          "pdf_only": "no LaTeX source (empty record)", "incomplete": "incomplete (a version missing)"}
counts = pd.crosstab(papers["range"], papers["outcome"]).reindex(columns=order, fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.9), gridspec_kw={"width_ratios": [3, 2]})
for ax, columns, title in ((axes[0], order, "All assigned IDs"),
                           (axes[1], order[1:], "Without the 'LaTeX for every version' papers")):
    left = np.zeros(len(counts))
    for name in columns:
        ax.barh(counts.index, counts[name], left=left, color=rd.SERIES[order.index(name)], label=labels[name],
                height=0.6, edgecolor=rd.SURFACE, linewidth=1.5)
        left += counts[name].to_numpy()
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Papers")
    ax.set_title(title)
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.1. Paper-level outcome by ID range. Right: the same bars without the dominant "
                  "class, to make the three small classes visible.")

### Version-level outcomes

Paper-level success hides what happened to each version, so the same result is broken down per version.
The number of versions of each paper comes from entry discovery; a version is *processed* when it has a
receipt, and an unprocessed version is classified by the `error_kind` of its **last** failed request.

In [ ]:
expected = int(meta["versions"].sum())
status = versions["status"].value_counts()
done_keys = set(zip(versions["id"], versions["version"]))
failed = events[events["log"] == "errors"].sort_values("at").groupby(["id", "version"]).tail(1)
failed = failed[[key not in done_keys for key in zip(failed["id"], failed["version"])]]
unprocessed = failed["error_kind"].value_counts()
KIND_TEXT = {
    "NOT_FOUND": "HTTP 404/410: arXiv has no source for this version",
    "CORRUPT_ARCHIVE": "archive cannot be unpacked",
    "REJECTED_ARCHIVE": "archive rejected (unsafe path, special file or unrecognised single file)",
    "TRANSFER_CUT": "transfer cut and not recovered",
}
rows = [("processed: `.tex`/`.bib` kept (`ok`)", status.get("ok", 0)),
        ("processed: no LaTeX source, empty record (`empty_source`)", status.get("empty_source", 0))]
rows += [(f"not processed: {KIND_TEXT.get(kind, kind)} (`{kind}`)", n) for kind, n in unprocessed.items()]
logged = sum(n for _, n in rows)
if expected - logged:
    rows.append(("not processed: no request logged", expected - logged))
t32 = pd.DataFrame(rows, columns=["Outcome of the version", "Versions"])
t32["Share"] = 100 * t32["Versions"] / expected
t32 = pd.concat([t32, pd.DataFrame([("**Versions reported by entry discovery**", expected, 100.0)],
                                   columns=t32.columns)], ignore_index=True)
rd.table(t32, "Table 3.2. Outcome of every source version.", {"Share": "{:.2f}%"})

missing = expected - len(versions)
not_found = int(unprocessed.get("NOT_FOUND", 0))
latest = meta.set_index("id")["versions"]
is_latest = sum(latest[i] == v for i, v in zip(failed["id"], failed["version"]))
rd.md(
    f"**{len(versions):,}** of the **{expected:,}** versions were processed ({pct(len(versions), expected)}); "
    f"**{status.get('empty_source', 0):,}** of them ({pct(status.get('empty_source', 0), len(versions))}) have no "
    f"LaTeX source. The **{missing:,}** unprocessed versions belong to the {int(total['Incomplete']):,} incomplete "
    f"papers. For **{not_found:,}** of them `export.arxiv.org/src/` answers HTTP 404 although the arXiv API lists "
    f"the version ({is_latest:,} of the {len(failed):,} unprocessed versions are the latest version of their "
    f"paper, the others are earlier ones). A 404 is not retried within a run, but these versions were asked again "
    f"in every later run, the last time on {failed['at'].max():%d %b %Y}, with the same answer; the source of "
    f"such a version is not available from the mirror (for instance a withdrawn version), so the scraper cannot "
    f"do better than record it. The remaining {missing - not_found:,} versions are archives the extractor "
    f"refuses or cannot read.")

### Failed and refused requests

`errors.jsonl` records every request that **failed** (the version is asked again in the next run) and
`throttles.jsonl` every request arXiv **refused** or left unanswered (the version goes back to the end of
the queue). These are *events*, not versions: one version can fail several times before it succeeds.

In [ ]:
ev = events.copy()
ev["recovered"] = [key in done_keys for key in zip(ev["id"], ev["version"])]
by_kind = ev.groupby("error_kind").agg(events=("id", "size"),
                                       versions=("id", lambda s: len(set(zip(s, ev.loc[s.index, "version"])))),
                                       ).sort_values("events", ascending=True)
recovered = ev[ev["recovered"]].groupby("error_kind").apply(
    lambda d: len(set(zip(d["id"], d["version"]))), include_groups=False)
by_kind["recovered"] = recovered.reindex(by_kind.index).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(8.5, 0.5 * len(by_kind) + 1.2))
bars = ax.barh(by_kind.index, by_kind["events"], color=rd.BLUE, height=0.6)
ax.bar_label(bars, labels=[f"{e:,} events · {v:,} versions, {r:,} later processed"
                           for e, v, r in zip(by_kind["events"], by_kind["versions"], by_kind["recovered"])],
             padding=4, color="#52514e", fontsize=9)
ax.set_xlim(0, by_kind["events"].max() * 1.75)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Logged events")
ax.set_title("Failed and refused source requests by error kind")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.2. Events in `errors.jsonl` and `throttles.jsonl` by `error_kind`, with the number "
                  "of distinct versions concerned and how many of those were processed in the end. Lines "
                  "written before the label existed are labelled from their exception text by the export script.")

n_versions = len(set(zip(ev["id"], ev["version"])))
n_recovered = len(set(zip(ev.loc[ev["recovered"], "id"], ev.loc[ev["recovered"], "version"])))
cut = by_kind.loc["TRANSFER_CUT"] if "TRANSFER_CUT" in by_kind.index else None
throttle = int(ev["error_kind"].isin(["HTTP_429", "HTTP_5XX"]).sum())
rd.md(
    f"The logs hold **{len(ev):,}** events for **{n_versions:,}** distinct versions "
    f"({pct(n_versions, expected)} of all versions); **{n_recovered:,}** of those versions were processed by a "
    f"later request, which is what the resume logic is for. "
    + (f"Transfers cut by the server (`TRANSFER_CUT`) produced {int(cut['events']):,} events on "
       f"{int(cut['versions']):,} versions, of which {int(cut['recovered']):,} succeeded later. " if cut is not None else "")
    + f"Rate limiting and server errors (`HTTP_429`, `HTTP_5XX`) account for {throttle:,} events. The versions "
      f"still unprocessed at the end are those of Table 3.2.")

### Check of the stored corpus

A receipt is written only after the files of a version are renamed into place, and the pipeline trusts the
receipts when it resumes (`DOWNLOAD_VERIFY = "receipt"`). Because Google Drive syncs asynchronously, the
export script also checks that the folder of every `ok` receipt really exists in the stored corpus.

In [ ]:
check = rd.corpus_check()
if check["ok_receipts_without_folder"]:
    rd.md(
        f"When the report data was exported ({check['checked_at'][:10]}), **{check['ok_receipts_without_folder']:,}** "
        f"of the {check['ok_receipts']:,} `ok` receipts ({pct(check['ok_receipts_without_folder'], check['ok_receipts'])}) "
        f"had **no `vN/` folder** in `data/raw/`: {check['papers_affected']:,} papers, all between "
        f"`{check['first_id']}` and `{check['last_id']}`. The receipts in `state/` were synced but the files of "
        f"that stretch were not, most likely because a Colab session ended before Drive finished syncing `data/`. "
        f"The versions were downloaded and measured (their sizes and times in this report are real), but their "
        f"files have to be downloaded again before Milestone 2: `src/09_repair_missing_sources.ipynb` finds these "
        f"versions and fetches them with the unchanged downloader. The success rate above follows the receipts; "
        f"counting only versions whose files are in place gives "
        f"**{pct(len(versions) - check['ok_receipts_without_folder'], expected)}** of the versions until the repair "
        f"has run.")
else:
    rd.md(f"When the report data was exported ({check['checked_at'][:10]}), the folder of every one of the "
          f"{check['ok_receipts']:,} `ok` receipts was present in `data/raw/`.")

## 3.2 Paper size before and after figure removal

**What is measured.** For every version, the receipt stores `before_bytes`, the size of **all files of the
unpacked source archive**, and `after_bytes`, the size of the `.tex` and `.bib` files that are kept. A
paper's size is the sum over its versions. Both are sizes of file contents, not of the compressed archive
and not of disk blocks. Sizes use binary units: 1 MB = 2²⁰ bytes, 1 GB = 2³⁰ bytes.

**Population.** Papers that were scraped successfully and have LaTeX source for at least one version, the
same population as `avg_paper_bytes_before/after` in `benchmark.json`. Papers without any LaTeX source have
nothing to compare and are left out rather than counted as a 0% reduction.

For paper $i$ with sizes $B_i$ (before) and $A_i$ (after), two reductions are reported, because they
answer different questions:

$$
R_i = 1 - \frac{A_i}{B_i} \quad\text{(per paper)}, \qquad
R_{\text{total}} = 1 - \frac{\sum_i A_i}{\sum_i B_i} \quad\text{(over all bytes)}.
$$

In [ ]:
sized = papers[papers["outcome"].isin(["latex", "mixed"])].copy()
sized["reduction"] = 1 - sized["size_after"] / sized["size_before"]
before, after = sized["size_before"], sized["size_after"]
assert abs(before.mean() - bench["avg_paper_bytes_before"]) < 1, "papers.csv and benchmark.json disagree"


def size_row(name, func, unit, scale):
    return (f"{name} ({unit})", func(before) / scale, func(after) / scale)


t33 = pd.DataFrame([
    size_row("Total", np.sum, "GB", rd.GB),
    size_row("Mean per paper", np.mean, "MB", rd.MB),
    size_row("Median per paper", np.median, "MB", rd.MB),
    size_row("95th percentile", lambda s: s.quantile(0.95), "MB", rd.MB),
    size_row("Maximum", np.max, "MB", rd.MB),
], columns=["Statistic", "Before (all files)", "After (.tex and .bib only)"])
rd.table(t33, f"Table 3.3. Paper size before and after figure removal, over {len(sized):,} papers with LaTeX source.",
         {"Before (all files)": "{:,.3f}", "After (.tex and .bib only)": "{:,.3f}"})

total_reduction = 1 - after.sum() / before.sum()
rd.md(
    f"The **average paper shrinks from {before.mean() / rd.MB:.2f} MB to {after.mean() / rd.MB:.3f} MB** "
    f"({before.mean():,.0f} → {after.mean():,.0f} bytes). Over all bytes the reduction is "
    f"**{total_reduction:.2%}** ({before.sum() / rd.GB:.1f} GB → {after.sum() / rd.GB:.2f} GB). "
    f"The typical paper saves a little less, a median of **{sized['reduction'].median():.1%}**, and the mean of the "
    f"per-paper reductions is only {sized['reduction'].mean():.1%}: large archives full of figures dominate the "
    f"total, while many small papers have little to remove. The sizes are strongly right-skewed (mean before "
    f"{before.mean() / before.median():.1f}× the median), so the mean alone does not describe a typical paper.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))
bins = np.logspace(2.5, np.log10(before.max()) + 0.05, 70)
ax = axes[0]
ax.hist(before, bins=bins, histtype="stepfilled", color=rd.ORANGE, alpha=0.30, edgecolor=rd.ORANGE, linewidth=1.6,
        label="before (all files)")
ax.hist(after.clip(lower=10 ** 2.5), bins=bins, histtype="stepfilled", color=rd.BLUE, alpha=0.30, edgecolor=rd.BLUE,
        linewidth=1.6, label="after (.tex/.bib)")
ax.set_xscale("log")
ax.set_xticks([1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9], ["1 kB", "10 kB", "100 kB", "1 MB", "10 MB", "100 MB", "1 GB"])
ax.set_xlabel("Paper size, all versions (log scale)")
ax.set_ylabel("Papers")
ax.set_title("Size distribution")
ax.legend(loc="upper right")

ax = axes[1]
reduction = np.sort(sized["reduction"].to_numpy()) * 100
ax.plot(reduction, np.arange(1, len(reduction) + 1) / len(reduction) * 100, color=rd.BLUE)
for level in (50, 90):
    share = (reduction < level).mean() * 100
    ax.plot([level], [share], "o", color=rd.BLUE, markersize=6, markeredgecolor=rd.SURFACE)
    ax.annotate(f"{share:.0f}% of papers save\nless than {level}%", (level, share), xytext=(-8, 8),
                textcoords="offset points", ha="right", fontsize=8.5, color="#52514e")
ax.set_xlabel("Size reduction of the paper (%)")
ax.set_ylabel("Papers with at most this reduction (%)")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_title("Cumulative distribution of the reduction")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.3. Left: paper size before and after keeping only `.tex`/`.bib` (logarithmic axis). "
                  "Right: cumulative distribution of the per-paper reduction.")


def top_share(series, fraction):
    ranked = series.sort_values(ascending=False)
    return ranked.iloc[: max(1, int(len(ranked) * fraction))].sum() / ranked.sum()


largest = sized.loc[before.idxmax()]
rd.md(
    f"**{(sized['reduction'] >= 0.9).mean():.1%}** of the papers lose at least 90% of their size and "
    f"{(sized['reduction'] >= 0.5).mean():.1%} at least half, but **{(sized['reduction'] < 0.1).mean():.1%}** lose "
    f"less than 10%: their archive holds little besides the LaTeX text. Storage is concentrated in few papers: "
    f"the largest 1%, 5% and 10% of the papers hold {top_share(before, 0.01):.1%}, {top_share(before, 0.05):.1%} "
    f"and {top_share(before, 0.10):.1%} of the bytes before filtering, and still {top_share(after, 0.01):.1%}, "
    f"{top_share(after, 0.05):.1%} and {top_share(after, 0.10):.1%} of the bytes that are kept. The largest "
    f"paper, `{largest['id']}`, goes from {largest['size_before'] / rd.MB:,.0f} MB to "
    f"{largest['size_after'] / rd.MB:.2f} MB.")

In [ ]:
t34 = sized.groupby("range").agg(papers=("id", "size"), before=("size_before", "sum"), after=("size_after", "sum"),
                                 median_before=("size_before", "median"), median_after=("size_after", "median"),
                                 median_reduction=("reduction", "median")).reset_index()
t34["total_reduction"] = 100 * (1 - t34["after"] / t34["before"])
t34["before"] /= rd.GB
t34["after"] /= rd.GB
t34["median_before"] /= rd.MB
t34["median_after"] /= rd.MB
t34["median_reduction"] *= 100
t34 = t34[["range", "papers", "before", "after", "total_reduction", "median_before", "median_after", "median_reduction"]]
t34.columns = ["Range", "Papers", "Before (GB)", "After (GB)", "Reduction over all bytes", "Median before (MB)",
               "Median after (MB)", "Median per-paper reduction"]
rd.table(t34, "Table 3.4. Size before and after by ID range (papers with LaTeX source).",
         {"Before (GB)": "{:.2f}", "After (GB)": "{:.3f}", "Reduction over all bytes": "{:.2f}%",
          "Median before (MB)": "{:.3f}", "Median after (MB)": "{:.3f}", "Median per-paper reduction": "{:.1f}%"})
rd.md(
    "The reduction is stable across the ranges. **What the numbers do and do not say:** the kept set is defined "
    "by file extension, so the bytes removed are figures *and* every other non-`.tex`/`.bib` file (`.bbl`, `.sty`, "
    "`.cls`, data files); the figures are by far the largest part, but the difference is the effect of the "
    "filtering rule as a whole. The `.tex` files are stored byte for byte, so `figure` environments and "
    "`\\includegraphics` commands are still in the text.")

## 3.3 Reference metadata

References come from the Semantic Scholar batch API, one list per paper (Semantic Scholar does not
distinguish arXiv versions). A lookup has three possible final outcomes, read from the `status` of each
paper's `references.json`:

- `ok`: Semantic Scholar knows the paper and returned its reference list, **which may be empty**;
- `not_found`: Semantic Scholar does not index the paper;
- `error`: the request kept failing.

The **reference success rate** is the share of papers with status `ok`. An empty list is reported
separately: it means Semantic Scholar returned no references, not that the paper cites nothing.

In [ ]:
fields = rd.reference_fields()
ref_log = rd.reference_log()
paper_status = fields["paper_status"]
ok_papers = paper_status.get("ok", 0)
n_papers = fields["papers"]
refs = papers.set_index("id")["n_refs"].fillna(0)
arxiv_refs = papers.set_index("id")["n_arxiv_refs"].fillna(0)
final_status = ref_log.sort_values("logged_at").groupby("id")["status"].last()
not_ok = set(final_status[final_status == "not_found"].index)
is_ok = ~papers["id"].isin(not_ok) if paper_status.get("not_found", 0) == len(not_ok) else papers["n_refs"].notna()
ok = papers[is_ok]
empty = int((ok["n_refs"].fillna(0) == 0).sum())

rows = [("`ok`, at least one reference", ok_papers - empty), ("`ok`, empty reference list", empty),
        ("`not_found` (not indexed by Semantic Scholar)", paper_status.get("not_found", 0)),
        ("`error` or file missing", n_papers - ok_papers - paper_status.get("not_found", 0))]
t35 = pd.DataFrame(rows, columns=["Final outcome", "Papers"])
t35["Share of assigned IDs"] = 100 * t35["Papers"] / n_papers
rd.table(t35, f"Table 3.5. Final outcome of the reference lookup over {n_papers:,} papers.",
         {"Share of assigned IDs": "{:.2f}%"})

assert ok_papers == bench["references_ok"], "references.json files and benchmark.json disagree"
n = ok["n_refs"].fillna(0)
rd.md(
    f"The **reference success rate is {pct(ok_papers, n_papers)}** ({ok_papers:,} papers). Those papers have "
    f"**{fields['references_total']:,}** references in total, **{n.mean():.2f} per paper** on average (median "
    f"{n.median():.0f}, interquartile range {n.quantile(0.25):.0f}–{n.quantile(0.75):.0f}, 95th percentile "
    f"{n.quantile(0.95):.0f}, maximum {n.max():.0f}). The average counts the {empty:,} papers with an empty list "
    f"as zero; over the papers with at least one reference it is {n[n > 0].mean():.2f}. A reference cited by "
    f"several papers is counted once per citing paper, so the total is a number of citations, not of distinct "
    f"works.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2), sharey=True)
cap = int(n.quantile(0.99))
bins = np.arange(0, cap + 3, 2)
for ax, values, color, title in ((axes[0], n, rd.BLUE, "All references"),
                                 (axes[1], ok["n_arxiv_refs"].fillna(0), rd.AQUA, "References with an arXiv ID")):
    ax.hist(values.clip(upper=cap), bins=bins, color=color, edgecolor=rd.SURFACE, linewidth=0.4)
    ax.axvline(values.mean(), color=rd.INK, linewidth=1, linestyle="--")
    ax.annotate(f"mean {values.mean():.1f}", (values.mean(), 1), xycoords=("data", "axes fraction"),
                xytext=(5, -12), textcoords="offset points", fontsize=8.5, color="#52514e")
    ax.set_xlabel(f"References per paper (last bin: {cap} or more)")
    ax.set_title(title)
axes[0].set_ylabel("Papers")
plt.tight_layout()
plt.show()
rd.figure_caption(f"Figure 3.4. References per paper over the {ok_papers:,} papers with status `ok`: all references "
                  "(left) and those that carry an arXiv ID (right). The first bar includes the empty lists.")

In [ ]:
t36 = ok.groupby("range").agg(papers=("id", "size"), empty=("n_refs", lambda s: int((s.fillna(0) == 0).sum())),
                              refs=("n_refs", "sum"), arxiv=("n_arxiv_refs", "sum"),
                              median=("n_refs", "median")).reset_index()
t36["mean"] = t36["refs"] / t36["papers"]
t36["arxiv_mean"] = t36["arxiv"] / t36["papers"]
t36["arxiv_share"] = 100 * t36["arxiv"] / t36["refs"]
t36["empty_share"] = 100 * t36["empty"] / t36["papers"]
t36 = t36[["range", "papers", "empty_share", "mean", "median", "arxiv_mean", "arxiv_share"]]
t36.columns = ["Range", "Papers `ok`", "Empty lists", "Mean references", "Median references",
               "Mean arXiv references", "arXiv share of references"]
rd.table(t36, "Table 3.6. Reference profile by ID range (papers with status `ok`).",
         {"Empty lists": "{:.1f}%", "Mean references": "{:.2f}", "Median references": "{:.0f}",
          "Mean arXiv references": "{:.2f}", "arXiv share of references": "{:.1f}%"})

total_refs = fields["references_total"]
present = fields["fields_present"]
rd.md(
    f"**{pct(present['ref_arxiv_id'], total_refs, 1)}** of the references carry an arXiv ID "
    f"({present['ref_arxiv_id']:,} references, **{arxiv_refs.sum() / ok_papers:.2f} per paper**); only these become "
    f"rows of the `references` table, the candidate pool for the citation matching of Milestone 3. "
    f"{present['arxiv_id_old_style']:,} of them use an old-style identifier such as `hep-th/9711200`, which is kept "
    f"in its original form. All references, with or without an arXiv ID, stay in `references.json`. "
    f"The range submitted most recently before the crawl (R1) has the most empty lists and the fewest references "
    f"per paper, which is what the indexing delay of Semantic Scholar would produce; the data does not prove "
    f"that cause.")

### Completeness of the reference fields

The statement asks for the title, authors, submission date, arXiv ID and Semantic Scholar ID of each
reference. The table counts, over **all saved references**, how often each field is present. A present
field is not checked for correctness.

In [ ]:
names = {"ref_title": "`ref_title`", "ref_authors": "`ref_authors` (at least one author)",
         "ref_submission_date": "`ref_submission_date`", "ref_s2_id": "`ref_s2_id`",
         "ref_arxiv_id": "`ref_arxiv_id`", "doi": "DOI (kept in `external_ids`)", "venue": "venue"}
t37 = pd.DataFrame([(label, present[key], total_refs - present[key]) for key, label in names.items()],
                   columns=["Field", "Present", "Missing"])
t37["Completeness"] = 100 * t37["Present"] / total_refs
rd.table(t37, f"Table 3.7. Field completeness over {total_refs:,} saved references.", {"Completeness": "{:.2f}%"})

source = fields["date_source"]
rd.md(
    f"Every saved reference has a title. **{pct(total_refs - present['ref_s2_id'], total_refs, 1)}** have no "
    f"Semantic Scholar ID: Semantic Scholar returns them as a citation string that it did not link to a paper "
    f"record, and most of them have no author list either. They cannot be matched by identifier but are kept, "
    f"because their title is still usable. The submission date comes from `publicationDate` for "
    f"{pct(source.get('publicationDate', 0), total_refs, 1)} of the references, falls back to the publication "
    f"`year` for {pct(source.get('year', 0), total_refs, 1)} and is missing for "
    f"{pct(source.get('none', 0), total_refs, 1)} (`date_source` records which one was used). "
    f"{fields['references_dropped']:,} entries were dropped for having neither an ID nor a title.")

### What the request log shows

`logs/references.csv` gets one row per paper every time its references are requested, so it also records
the failed rounds that the final files no longer show.

In [ ]:
ref_log["hour"] = ref_log["logged_at"].dt.floor("h")
rounds = ref_log.groupby(["hour", "status"]).size().unstack(fill_value=0).reindex(
    columns=["ok", "not_found", "error"], fill_value=0).reset_index()
rounds["hour"] = rounds["hour"].dt.strftime("%Y-%m-%d %H:00 UTC")
rounds.columns = ["Hour", "`ok` rows", "`not_found` rows", "`error` rows"]
rd.table(rounds, "Table 3.8. Rows written to the reference log, by hour and status.")

errors = int((ref_log["status"] == "error").sum())
ok_time = ref_log.loc[ref_log["status"] == "ok", "seconds"]
rd.md(
    f"The log has {len(ref_log):,} rows for {ref_log['id'].nunique():,} papers. **{errors:,} rows are errors**, "
    f"all of them batches that Semantic Scholar refused with HTTP 429 eight times in a row, concentrated in the "
    f"first session. A failed batch never overwrites a saved file and is simply asked again, so the following "
    f"runs filled every gap: the final files contain no `error`. A successful request took "
    f"{ok_time.median():.2f} s per paper (median; the time of a batch of 100 is divided among its papers). "
    f"There are more `ok` rows than papers because the papers were requested again in later runs.")

## 3.4 Versions, subjects and dates

These breakdowns use the metadata collected by entry discovery (`metadata.csv`, one row per paper).

### Versions per paper

In [ ]:
dist = meta["versions"].value_counts().sort_index()
t39 = pd.DataFrame({"Versions": dist.index, "Papers": dist.values})
t39["Share of papers"] = 100 * t39["Papers"] / len(meta)
t39["Source versions"] = t39["Versions"] * t39["Papers"]

fig, ax = plt.subplots(figsize=(7.5, 2.9))
bars = ax.bar(dist.index.astype(str), dist.values, color=rd.BLUE, width=0.62)
ax.bar_label(bars, labels=[f"{v:,}" for v in dist.values], padding=2, color="#52514e", fontsize=9)
ax.set_xlabel("Number of versions")
ax.set_ylabel("Papers")
ax.set_ylim(0, dist.max() * 1.12)
ax.grid(axis="x", visible=False)
ax.set_title("Versions per paper")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.5. Number of versions per paper.")
rd.table(t39, "Table 3.9. Distribution of the number of versions.", {"Share of papers": "{:.2f}%"})

multi = meta[meta["versions"] > 1]
rd.md(
    f"The {len(meta):,} papers have **{expected:,} versions**, {meta['versions'].mean():.3f} per paper. "
    f"**{pct((meta['versions'] == 1).sum(), len(meta), 1)}** of the papers have a single version; the "
    f"**{len(multi):,}** multi-version papers ({pct(len(multi), len(meta), 1)}) account for "
    f"{pct(multi['versions'].sum(), expected, 1)} of all versions, and the maximum is "
    f"{meta['versions'].max()} versions. Crawling every version instead of only the latest one therefore means "
    f"**{expected - len(meta):,} additional source requests** ({pct(expected - len(meta), len(meta), 1)} more).")

In [ ]:
meta["interval"] = (meta["last_revised"] - meta["submitted"]).dt.days
t310 = meta.groupby("range").agg(versions=("versions", "sum"), multi=("versions", lambda s: int((s > 1).sum())),
                                 share=("versions", lambda s: 100 * (s > 1).mean()),
                                 first=("submitted", "min"), last=("submitted", "max"),
                                 interval=("interval", "median")).reset_index()
t310["range"] = t310["range"].map(RANGE_LABEL)
t310["first"] = t310["first"].dt.strftime("%Y-%m-%d")
t310["last"] = t310["last"].dt.strftime("%Y-%m-%d")
t310.columns = ["Range", "Versions", "Multi-version papers", "Multi-version share", "Earliest submission",
                "Latest submission", "Median days to latest revision"]
rd.table(t310, "Table 3.10. Versions and revisions by ID range. The revision interval is from the first "
               "submission to the latest version, over multi-version papers only.",
         {"Multi-version share": "{:.1f}%", "Median days to latest revision": "{:.1f}"})
rd.md(
    "The older a range, the more of its papers have been revised and the longer the interval to the latest "
    "revision. This reflects how long each range had been online when it was crawled (early November 2026), "
    "not a difference in how authors revise: the September papers have had little time to receive a second "
    "version. The intervals are to the latest version *observed*, and more revisions may still come.")

### Subjects

Each paper is counted once, under its **primary category** (the first category in its arXiv metadata);
cross-listings are ignored.

In [ ]:
cats = meta["primary_category"].value_counts()
top_n = 15
top = cats.head(top_n)
shown = pd.concat([top, pd.Series({f"other ({len(cats) - top_n} categories)": cats.iloc[top_n:].sum()})])
fig, ax = plt.subplots(figsize=(8.5, 4.6))
colors = [rd.BLUE] * top_n + [rd.AXIS]
bars = ax.barh(shown.index[::-1], shown.values[::-1], color=colors[::-1], height=0.66)
ax.bar_label(bars, labels=[f"{v:,}  ({v / len(meta):.1%})" for v in shown.values[::-1]], padding=4,
             color="#52514e", fontsize=9)
ax.set_xlim(0, shown.max() * 1.18)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Papers")
ax.set_title(f"Primary category: the {top_n} largest and the rest")
plt.tight_layout()
plt.show()
rd.figure_caption(f"Figure 3.6. Papers by primary arXiv category. The grey bar groups the other categories so "
                  "that the chart covers the whole corpus.")

archive = meta["primary_category"].str.split(".").str[0].value_counts()
rd.md(
    f"The corpus spans **{len(cats)} primary categories**. The four largest, "
    f"{', '.join(f'`{c}`' for c in cats.index[:4])}, hold {pct(cats.iloc[:4].sum(), len(meta), 1)} of the papers, "
    f"and the {top_n} largest {pct(top.sum(), len(meta), 1)}. By archive, `{archive.index[0]}` has "
    f"{pct(archive.iloc[0], len(meta), 1)} of the papers and `{archive.index[1]}` {pct(archive.iloc[1], len(meta), 1)}. "
    f"The mix matters for the next milestones: the long tail of {len(cats) - top_n} smaller categories means the parser will meet many "
    f"different LaTeX styles.")

In [ ]:
heat = pd.crosstab(meta["primary_category"], meta["range"]).loc[cats.index[:10]]
heat = 100 * heat / meta.groupby("range").size()
fig, ax = plt.subplots(figsize=(6.2, 3.9))
image = ax.imshow(heat.to_numpy(), cmap="Blues", vmin=0, aspect="auto")
ax.set_xticks(range(heat.shape[1]), heat.columns)
ax.set_yticks(range(heat.shape[0]), heat.index)
ax.grid(False)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
threshold = heat.to_numpy().max() * 0.6
for (row, column), value in np.ndenumerate(heat.to_numpy()):
    ax.text(column, row, f"{value:.1f}", ha="center", va="center", fontsize=9,
            color="#ffffff" if value > threshold else rd.INK)
ax.set_title("Share of each range's papers (%)")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.7. The ten largest primary categories as a percentage of the 5,000 papers of each "
                  "range. The ranges are fixed ID windows, not random samples of arXiv.")

### Submission dates

An arXiv ID starts with the year and month in which the identifier was issued (`2609` = September 2026).
The table compares that prefix with the month of the **recorded submission date of version 1**.

In [ ]:
meta["id_month"] = "20" + meta["id"].str[:2] + "-" + meta["id"].str[2:4]
meta["submitted_month"] = meta["submitted"].dt.strftime("%Y-%m")
months = pd.crosstab(meta["submitted_month"], meta["range"])
months["All"] = months.sum(axis=1)
months = months.reset_index().rename(columns={"submitted_month": "Month of first submission"})
rd.table(months, "Table 3.11. Papers by month of first submission and ID range.")

earlier = int((meta["submitted_month"] < meta["id_month"]).sum())
later = int((meta["submitted_month"] > meta["id_month"]).sum())
venue = int(meta["has_venue"].sum())
rd.md(
    f"**{pct(len(meta) - earlier - later, len(meta))}** of the papers were first submitted in the month of their "
    f"ID prefix. **{earlier:,}** ({pct(earlier, len(meta))}) have an earlier recorded date and {later:,} a later "
    f"one, so the ID prefix is not a substitute for the submission date; the `papers` table uses the recorded "
    f"date of version 1. A journal reference (`venue`) is present for only **{pct(venue, len(meta), 1)}** of the "
    f"papers and a DOI for {pct(meta['has_doi'].sum(), len(meta), 1)}, as expected for preprints a few months "
    f"old, which is why `venue` is nullable in the `papers` table.")

In [ ]:
interval = meta["interval"].dropna()
fig, ax = plt.subplots(figsize=(8.5, 3))
ax.hist(interval, bins=np.arange(0, interval.max() + 8, 7), color=rd.BLUE, edgecolor=rd.SURFACE, linewidth=0.4)
ax.axvline(interval.median(), color=rd.INK, linewidth=1, linestyle="--")
ax.annotate(f"median {interval.median():.0f} days", (interval.median(), 1), xycoords=("data", "axes fraction"),
            xytext=(5, -12), textcoords="offset points", fontsize=8.5, color="#52514e")
ax.set_xlabel("Days from first submission to latest version (one bar per week)")
ax.set_ylabel("Multi-version papers")
ax.set_title("Time to the latest revision")
plt.tight_layout()
plt.show()
rd.figure_caption(f"Figure 3.8. Days between the first submission and the latest version, for the {len(interval):,} "
                  "multi-version papers.")
rd.md(
    f"Among revised papers the median time to the latest version is **{interval.median():.0f} days** (mean "
    f"{interval.mean():.1f}, maximum {interval.max():.0f}); {pct((interval <= 30).sum(), len(interval), 1)} were "
    f"last revised within 30 days. The shape mixes the four ranges of Table 3.10, each with a different time "
    f"online, so it describes this corpus rather than revision behaviour in general.")